# Aurora SIGER — Fase 1: Telemetria e Deteccao de Anomalias

> Sistema Inteligente de Gerenciamento de Riscos para telemetria de decolagem espacial.

Este notebook demonstra o pipeline completo da Fase 1:
1. Geracao de dataset sintetico de telemetria (100k amostras)
2. Analise exploratoria dos dados
3. Isolation Forest implementado do zero vs. Scikit-learn
4. Pipeline de decisao GO/NO-GO

In [ ]:
# Install package in editable mode (run once)
# !pip install -e ../..['viz']

from aurora_siger.data.generation import generate_telemetry_dataset
from aurora_siger.eda.plots import (
    configure_style,
    heatmap_plot,
    distribution_plot,
    boxplot_analysis,
    pairplot_data,
    scatter_3d_anomaly,
)
from aurora_siger.models.isolation_forest import MyIsolationForest
from aurora_siger.pipeline.validator import Validator
from aurora_siger.pipeline.launch import (
    ai_anomaly_check,
    calculate_autonomy,
    launch_decision,
)

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
import matplotlib.pyplot as plt

configure_style()

## 1. Geracao do Dataset

100.000 amostras sinteticas de telemetria com 7 sensores.
3% de anomalias com distribuicoes deliberadamente deslocadas.

In [ ]:
df = generate_telemetry_dataset(n_samples=100_000, anomaly_ratio=0.03, seed=42)
print(f"Shape: {df.shape}")
print(f"Anomalias: {df['anomaly'].sum()} ({df['anomaly'].mean():.1%})")
df.head()

In [ ]:
df.describe()

## 2. Analise Exploratoria (EDA)

In [ ]:
heatmap_plot(df)

In [ ]:
pairplot_data(df)

In [ ]:
boxplot_analysis(df)

In [ ]:
distribution_plot(df)

In [ ]:
scatter_3d_anomaly(df.sample(12_000, random_state=42), "tank_pressure", "internal_temp", "vibration")

## 3. Isolation Forest — Do Zero vs. Scikit-learn

Treinamento com split estratificado 80/20, escalonamento via StandardScaler.

In [ ]:
X = df.drop(columns=["anomaly"]).values
y = df["anomaly"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
my_model = MyIsolationForest(n_trees=100, sample_size=256)
my_model.fit(X_train)

train_scores = my_model.anomaly_score(X_train)
threshold = np.percentile(train_scores, 97)
test_scores = my_model.anomaly_score(X_test)
preds = np.where(test_scores >= threshold, 1, 0)

print("Scratch Model")
print(classification_report(y_test, preds))
scratch_auc = roc_auc_score(y_test, test_scores)
print(f"ROC AUC: {scratch_auc:.4f}")

In [ ]:
sk_model = IsolationForest(n_estimators=100, max_samples=256, contamination=0.03, random_state=42)
sk_model.fit(X_train)

sk_preds = np.where(sk_model.predict(X_test) == -1, 1, 0)
sk_scores = -sk_model.decision_function(X_test)

print("Sklearn Model")
print(classification_report(y_test, sk_preds))
sk_auc = roc_auc_score(y_test, sk_scores)
print(f"ROC AUC: {sk_auc:.4f}")

In [ ]:
plt.hist(test_scores[y_test == 0], bins=50, alpha=0.6, label="Normal")
plt.hist(test_scores[y_test == 1], bins=50, alpha=0.6, label="Anomaly")
plt.axvline(threshold, color="red", linestyle="--", label="Threshold")
plt.title("Isolation Forest Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]

scratch_scores_list = [
    accuracy_score(y_test, preds),
    precision_score(y_test, preds),
    recall_score(y_test, preds),
    f1_score(y_test, preds),
    scratch_auc,
]
sklearn_scores_list = [
    accuracy_score(y_test, sk_preds),
    precision_score(y_test, sk_preds),
    recall_score(y_test, sk_preds),
    f1_score(y_test, sk_preds),
    sk_auc,
]

x = np.arange(len(metrics))
width = 0.35
plt.figure(figsize=(8, 5))
plt.bar(x - width / 2, sklearn_scores_list, width, label="Sklearn")
plt.bar(x + width / 2, scratch_scores_list, width, label="Scratch")
plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Isolation Forest: Sklearn vs Scratch Implementation")
plt.legend()
plt.show()

## 4. Pipeline de Decisao GO/NO-GO

Teste com uma leitura normal e uma anomala.

In [ ]:
good_sample = {
    "internal_temp": 22.3, "external_temp": 12.0, "structural_integrity": 1,
    "energy": 98.0, "vibration": 0.32, "tank_pressure": 305.0, "critical_modules": 1,
}

bad_sample = {
    "internal_temp": 24.5, "external_temp": 110.0, "structural_integrity": 0,
    "energy": 63.0, "vibration": 0.45, "tank_pressure": 335.0, "critical_modules": 0,
}

for label, sample in [("GOOD", good_sample), ("BAD", bad_sample)]:
    print(f"\n{'='*50}")
    print(f"=== {label} SAMPLE ===")
    launch_decision(sample, my_model, scaler, threshold)